# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [ ]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


In [ ]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [2]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
# jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [ ]:
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
jugar_frozen_lake(env)

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [3]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 0


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [4]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 2.0%
Recompensa Promedio: 0.020
Desviación Estándar: 0.140
Total de Victorias: 2


La siguiente celda define una función para entrenar un agente.

In [5]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=100, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
            
            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")


✓ Funciones de entrenamiento definidas


## 1. **Agente Q-Learning**
La siguiente celda define el agente de Q-Learning a implementar.

In [ ]:
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self, espacio_observacion, espacio_acciones, alpha_offset=25, alpha_min=0.03,
                 gamma=0.99, epsilon=1, epsilon_decay=0.000015, epsilon_min=0.01):
        """
        Inicializa el agente Q-Learning.

        Args:
            espacio_observacion (int): Número de estados posibles.
            espacio_acciones (int): Número de acciones posibles.
            alpha_offset (float): Offset para el cálculo de alpha dinámico.
            alpha_min (float): Valor mínimo de alpha.
            gamma (float): Factor de descuento.
            epsilon (float): Probabilidad inicial de exploración (epsilon-greedy).
            epsilon_decay (float): Cantidad fija a disminuir epsilon por episodio.
        """
        self.qtable = np.zeros((espacio_observacion, espacio_acciones)) 
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.espacio_acciones = espacio_acciones
        # Para α dinámico
        self.visitas = np.zeros_like(self.qtable, dtype=np.int64)
        self.alpha_offset = alpha_offset
        self.alpha_min = alpha_min
    
    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        if random.uniform(0, 1) < self.epsilon:
            # Exploración: elige una acción aleatoria
            return random.randint(0, self.espacio_acciones - 1)
        else:
            # Explotación: elige la acción con el valor Q más alto para el estado actual
            return np.argmax(self.qtable[estado, :])
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando alpha dinámico basado en visitas."""
        # Actualizamos epsilon en cada paso
        self.epsilon = max(self.epsilon_min, self.epsilon - self.epsilon_decay)

        # Incrementar contador de visitas
        self.visitas[estado, accion] += 1
        # Calcular α dinámico: α(s,a) = max(α_min, 1/(offset + visitas))
        alpha_sa = max(self.alpha_min, 1.0 / (self.alpha_offset + self.visitas[estado, accion]))
        # Calcular target de TD
        if terminado:
            td_target = recompensa
        else:
            td_target = recompensa + self.gamma * np.max(self.qtable[siguiente_estado, :])
        # Actualizar Q-table con fórmula optimizada: Q = (1-α)*Q_old + α*target
        q_old = self.qtable[estado, accion]
        self.qtable[estado, accion] = (1.0 - alpha_sa) * q_old + alpha_sa * td_target
    
    def get_alpha_promedio(self):
        """Obtiene el valor promedio actual de alpha para análisis."""
        alphas = []
        for s in range(self.qtable.shape[0]):
            for a in range(self.qtable.shape[1]):
                if self.visitas[s, a] > 0:
                    alpha_sa = max(self.alpha_min, 1.0 / (self.alpha_offset + self.visitas[s, a]))
                    alphas.append(alpha_sa)
        return np.mean(alphas) if alphas else self.alpha_min
    
    def mostrar_hiperparametros(self):
        """Muestra los valores actuales de los hiperparámetros del agente."""
        print("Hiperparámetros del Agente Q-Learning")
        print(f"epsilon_decay   : {self.epsilon_decay}")
        print(f"gamma           : {self.gamma}")
        print(f"alpha offset    : {self.alpha_offset}")
        print(f"alpha min       : {self.alpha_min}")
        

## 2. **Agente DQN**
La siguiente celda define el agente DQN a implementar. 

In [ ]:
class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        # Conexiones capa de entrada a oculta
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto)
        # Conexiones capa oculta a salida
        self.fc2 = nn.Linear(tamano_oculto, tamano_salida)
    
    def forward(self, x):
        # Se aplica ReLU a la combinacion lineal que llega a la capa oculta
        x = F.relu(self.fc1(x))
        # Se retorna la combinacion lineal que llega a la capa de salida
        return self.fc2(x)

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""

    def __init__(self, n):
        """Inicializa el agente DQN.
        
        Args:
            observation_space_n (int): Número de estados posibles.
            action_space_n (int): Número de acciones posibles.
            alpha_offset (float): Offset para el cálculo de alpha dinámico.
            alpha_min (float): Valor mínimo de alpha.
            gamma (float): Factor de descuento.
            epsilon (float): Probabilidad inicial de exploración (epsilon-greedy).
            epsilon_decay (float): Cantidad fija a disminuir epsilon por episodio.
        """
        # Parametros de la red
        self.tamano_entrada = n*n
        self.tamano_salida = 4
        if n == 4:
            self.tamano_oculto = 128
            self.memoria_max = 10000
            self.batch_size = 64
            self.contador_tope = 1000
            self.lr = 0.00025
            self.epsilon_final = 0.01
            self.epsilon_decay = 0.85
            self.gamma = 0.99
        elif n == 8:
            self.tamano_oculto = 128
            self.memoria_max = 20000
            self.batch_size = 128
            self.contador_tope = 1000
            self.lr = 0.0002
            self.epsilon_final = 0.01
            self.epsilon_decay = 0.90
            self.gamma = 0.95
        self.epsilon = 1.0
        self.counter = self.contador_tope
        # Inicialización de la red
        self.model = DQN(self.tamano_entrada, self.tamano_oculto, self.tamano_salida)
        self.target_model = DQN(self.tamano_entrada, self.tamano_oculto, self.tamano_salida)
        self.target_model.load_state_dict(self.model.state_dict())
        self.target_model.eval()
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.memoria = deque(maxlen=self.memoria_max)

    def actualizar_red_objetivo(self):
        """Cada <self.contador> pasos actualiza los pesos de la red objetivo con los pesos de la red principal."""
        self.counter -= 1
        if self.counter == 0:
            self.target_model.load_state_dict(self.model.state_dict())
            self.counter = self.contador_tope

    def guardar_experiencia(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Guarda la experiencia (s, a, r, s', t) en la memoria del agente"""
        self.memoria.append((estado, accion, recompensa, siguiente_estado, terminado))
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        # Tomamos una acción aleatoria con probabilidad epsilon (exploración)
        if random.random() < self.epsilon:
            accion = np.random.randint(self.tamano_salida)
        # Tomamos la acción con mayor Q-valor con probabilidad 1 - epsilon (explotación)
        else:
            tensor_estado = torch.eye(self.tamano_entrada)[estado].float().unsqueeze(0)
            with torch.no_grad():
                q_valores = self.model(tensor_estado)
                accion = torch.argmax(q_valores).item()
        return accion
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la red neuronal usando la experiencia (s, a, r, s', t)."""
        # Aseguramos que este el grad_enabled (Por alguna razón falla si se cambia de 4x4 a 8x8 sin esta linea)
        torch.set_grad_enabled(True)

        # Si termino el episodio actualizamos epsilon
        if terminado:
            self.epsilon = max(self.epsilon_final, self.epsilon * self.epsilon_decay)

        # Empezamos guardando la experiencia en la memoria
        self.guardar_experiencia(estado, accion, recompensa, siguiente_estado, terminado)
        self.actualizar_red_objetivo()
        # Hasta no tener un batch completo en memoria no empezamos
        if len(self.memoria) < self.batch_size:
            return
        
        # 0 - Limpiamos los gradientes acumulados antes de la actualización
        self.optimizer.zero_grad()
        # 1 - Tomamos un batch aleatorio de la memoria y generamos tensores
        batch = random.sample(self.memoria, self.batch_size)
        estados, acciones, recompensas, siguientes_estados, terminados = zip(*batch)
        estados = torch.FloatTensor(np.identity(self.tamano_entrada)[list(estados)])
        acciones = torch.LongTensor(acciones).unsqueeze(1)
        recompensas = torch.FloatTensor(recompensas).unsqueeze(1)
        siguientes_estados = torch.FloatTensor(np.identity(self.tamano_entrada)[list(siguientes_estados)])
        terminados = torch.FloatTensor(terminados).unsqueeze(1)
        # 2 - Calculamos los Q-valores para estados actuales
        q_valores = self.model(estados)
        q_valor_accion = q_valores.gather(1, acciones)
        # 3 - Calculamos los objetivos (target) usando una red objetivo
        with torch.no_grad():
            q_valores_siguientes = self.target_model(siguientes_estados)
            max_q_valor_siguientes = q_valores_siguientes.max(1, keepdim=True)[0]
            objetivos = recompensas + (1 - terminados) * self.gamma * max_q_valor_siguientes
        # 4 - Calculamos la pérdida entre el Q-valores actuales y los objetivos
        perdida = self.criterion(q_valor_accion, objetivos)
        # 5 - Calculamos los gradientes mediante backpropagation
        perdida.backward()
        # 6 - Actualizamos los parámetros de la red usando el optimizador
        self.optimizer.step()

    def mostrar_hiperparametros(self):
        """Muestra los valores actuales de los hiperparámetros del agente."""
        print("Hiperparámetros del Agente DQN")
        print(f"tamaño_oculto   : {self.tamano_oculto}")
        print(f"memoria_max     : {self.memoria_max}")
        print(f"batch_size      : {self.batch_size}")
        print(f"contador_tope   : {self.contador_tope}")
        print(f"learning_rate   : {self.lr}")
        print(f"epsilon_final   : {self.epsilon_final}")
        print(f"epsilon_decay   : {self.epsilon_decay}")
        print(f"gamma           : {self.gamma}")


## 3. **Entrenamiento y Evaluación**
Las siguientes celdas definen funciones auxiliares para entrenar y evaluar los agentes

In [ ]:
def evaluar_agente_custom(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    Asegura que únicamente se hace explotación (seteando epsilon en 0).
    Al finalizar restaura epsilon.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    eps_backup = agente.epsilon
    # No nos interesa explorar en esta etapa
    agente.epsilon = 0.0
    resultados = evaluar_agente(agente, env, num_episodios=num_episodios)
    agente.epsilon = eps_backup
    return resultados

In [ ]:
import copy
import pandas as pd

# def entrenar_agente_bloques(cfg, episodios_total=150000, tam_bloque=15000, 
#                                       max_pasos=300, eval_intermedia=1000, epsilon_min=0.005,
#                                       alpha_offset=25, alpha_min=0.03):
def entrenar_agente_bloques(agente: AgenteQLearning, env, num_episodios=150000, eval_intermedia=1000,
                            max_pasos=100, tam_bloque=15000, verbose=True):
    """
    Versión de alpha modificada con:
    1. α(s,a) = max(α_min, 1/(offset + visitas)) - para que no caiga muy rápido
    2. ε_min más bajo (0.005) - más exploración tardía
    3. Más episodios y pasos por episodio
    """
    mejor_tasa = -1.0
    mejor_q = None
    progreso = []

    bloques = max(1, num_episodios // tam_bloque)
    if verbose:
        print(f"Entrenando {num_episodios} episodios en {bloques} bloques de {tam_bloque}")
        print(f"α dinámico: α(s,a) = max({agente.alpha_min}, 1/({agente.alpha_offset} + visitas))")
    
    for b in range(bloques):
        for ep in range(tam_bloque):
            estado, _ = env.reset()
            for paso in range(max_pasos):
                accion = agente.elegir_accion(estado)
                siguiente_estado, recompensa, terminado, truncado, _info = env.step(accion)

                agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)

                estado = siguiente_estado
                if terminado or truncado:
                    break

        # Evaluación y checkpoint
        res_mid = evaluar_agente_custom(agente, env, num_episodios=eval_intermedia)
        tasa_mid = res_mid['tasa_victorias']
        
        # Calcular α promedio actual
        alpha_promedio = agente.get_alpha_promedio()
        
        progreso.append({
            'bloque': b + 1, 'tasa': tasa_mid, 'epsilon': agente.epsilon, 
            'alpha_promedio': alpha_promedio, 'visitas_total': np.sum(agente.visitas)
        })
        if verbose:
            print(f"Bloque {b+1}/{bloques} | tasa_greedy={tasa_mid:.1%} | ε={agente.epsilon:.4f} | α_prom={alpha_promedio:.4f}")

        if tasa_mid > mejor_tasa:
            mejor_tasa = tasa_mid
            mejor_q = copy.deepcopy(agente.qtable)

    if mejor_q is not None:
        agente.qtable = mejor_q

    res_final = evaluar_agente_custom(agente, env, num_episodios=2000)

    return agente, res_final, pd.DataFrame(progreso)

In [ ]:
from itertools import product

def optimizar_hiperparametros(agente: Agente, param_ranges, env, num_episodios=1000, eval_episodios=1000):
    mejores_resultados = None
    mejor_agente = None
    param_grid = []

    # Generar combinaciones de hiperparámetros
    keys = param_ranges.keys()
    values = ( [round(float(v), 5) for v in param_ranges[k]] for k in keys )
    for combination in product(*values):
        params = dict(zip(keys, combination))
        param_grid.append(params)

    for params in param_grid:
        print(f"Probando configuración: {params}", flush=True)
        if isinstance(agente, AgenteDQN):
            n = int(agente.tamano_entrada**0.5)
            # Reiniciar agente DQN con tamaño adecuado
            agente = AgenteDQN(n)
            # Método get permite usar valores por defecto si no se especifican en params
            agente.tamano_oculto = params.get("tamano_oculto", agente.tamano_oculto)
            agente.memoria_max = params.get("memoria_max", agente.memoria_max)
            agente.batch_size = params.get("batch_size", agente.batch_size)
            agente.contador_tope = params.get("contador_tope", agente.contador_tope)
            agente.lr = params.get("lr", agente.lr)
            agente.epsilon_final = params.get("epsilon_final", agente.epsilon_final)
            agente.epsilon_decay = params.get("epsilon_decay", agente.epsilon_decay)
            agente.gamma = params.get("gamma", agente.gamma)
        # Evaluamos el agente con la configuración actual
        recompensas, _ = entrenar_agente(agente, env, num_episodios=20000)
        resultados = evaluar_agente(agente, env, num_episodios=eval_episodios)
        # Guardamos los mejores resultados
        if (mejores_resultados is None) or (resultados['recompensa_promedio'] > mejores_resultados['recompensa_promedio']):
            mejores_resultados = resultados
            mejor_agente = agente
        print(f"Recompensa promedio: {resultados['recompensa_promedio']}")

    print("\nMejor configuración encontrada:")
    mejor_agente.mostrar_hiperparametros()
    imprimir_resultados_evaluacion(mejores_resultados, "Optimización")
    return mejor_agente, mejores_resultados


In [ ]:
print("Estrategia α(s,a) = max(α_min, 1/(offset + visitas))")
print("="*60)

# Usamos otra version de alpha dinamico, ya que α=1/n decae muy rápido y "congela" valores Q.

def entrenar_alpha_visitas_optimizado(cfg, episodios_total=150000, tam_bloque=15000, 
                                      max_pasos=300, eval_intermedia=1000, epsilon_min=0.005,
                                      alpha_offset=25, alpha_min=0.03):
    """
    Versión de alpha modificada con:
    1. α(s,a) = max(α_min, 1/(offset + visitas)) - para que no caiga muy rápido
    2. ε_min más bajo (0.005) - más exploración tardía
    3. Más episodios y pasos por episodio
    """
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    agente = AgenteQLearning(
        observation_space_n=env.observation_space.n,
        action_space_n=env.action_space.n,
        alpha_offset=alpha_offset,
        alpha_min=alpha_min,
        gamma=cfg['gamma'],
        epsilon=1.0,
        epsilon_decay=cfg['epsilon_decay']
    )

    mejor_tasa = -1.0
    mejor_q = None
    progreso = []

    bloques = max(1, episodios_total // tam_bloque)
    print(f"Entrenando {episodios_total} episodios en {bloques} bloques de {tam_bloque}")
    print(f"α dinámico: α(s,a) = max({alpha_min}, 1/({alpha_offset} + visitas))")
    
    for b in range(bloques):
        for ep in range(tam_bloque):
            estado, _ = env.reset()
            for paso in range(max_pasos):
                accion = agente.elegir_accion(estado)
                siguiente_estado, recompensa, terminado, truncado, _info = env.step(accion)

                # Usar método de aprendizaje con alpha dinamico
                agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)

                estado = siguiente_estado
                if terminado or truncado:
                    break

            # Decaimiento más lento de epsilon
            agente.epsilon = max(epsilon_min, agente.epsilon - agente.epsilon_decay)

        # Evaluación y checkpoint
        res_mid = evaluar_greedy(agente, env, num_episodios=eval_intermedia, max_pasos=max_pasos)
        tasa_mid = res_mid['tasa_victorias']
        
        # Calcular α promedio actual
        alpha_promedio = agente.get_alpha_promedio()
        
        progreso.append({
            'bloque': b + 1, 'tasa': tasa_mid, 'epsilon': agente.epsilon, 
            'alpha_promedio': alpha_promedio, 'visitas_total': np.sum(agente.visitas)
        })
        print(f"Bloque {b+1}/{bloques} | tasa_greedy={tasa_mid:.1%} | ε={agente.epsilon:.4f} | α_prom={alpha_promedio:.4f}")

        if tasa_mid > mejor_tasa:
            mejor_tasa = tasa_mid
            mejor_q = copy.deepcopy(agente.qtable)

    if mejor_q is not None:
        agente.qtable = mejor_q

    res_final = evaluar_greedy(agente, env, num_episodios=2000, max_pasos=max_pasos)
    env.close()
    return agente, res_final, pd.DataFrame(progreso)


# Configuraciones optimizadas para α=1/visitas
configs_optimizadas = [
    {
        'nombre': 'Ultra_A2: γ=0.998, exploración extendida',
        'alpha': 0.1, 'gamma': 0.998,
        'epsilon_decay': (1.0 - 0.005) / (100000 * 0.6),  # 60% exploración
        'alpha_offset': 15, 'alpha_min': 0.04
    },
    {
        'nombre': 'Ultra_C2: γ=0.999, conservador',
        'alpha': 0.1, 'gamma': 0.999,
        'epsilon_decay': (1.0 - 0.003) / (100000 * 0.7),  # 70% exploración
        'alpha_offset': 25, 'alpha_min': 0.05
    },
    {
        'nombre': 'Ultra_Mix: γ=0.997, balance',
        'alpha': 0.1, 'gamma': 0.997,
        'epsilon_decay': (1.0 - 0.004) / (100000 * 0.65),  # 65% exploración
        'alpha_offset': 20, 'alpha_min': 0.035
    }
]

print("\n🚀 Probando configuraciones optimizadas...")
resultados_optimizacion = []

for i, cfg in enumerate(configs_optimizadas):
    print(f"\n🧪 Configuración {i+1}: {cfg['nombre']}")
    print(f"   γ={cfg['gamma']}, α_offset={cfg['alpha_offset']}, α_min={cfg['alpha_min']}")
    
    # Crear configuración compatible (sin alpha fijo)
    cfg_compatible = {
        'gamma': cfg['gamma'], 
        'epsilon_decay': cfg['epsilon_decay']
    }
    
    t0 = time.time()
    agent_opt, res_opt, prog_opt = entrenar_alpha_visitas_optimizado(
        cfg_compatible,
        episodios_total=100000,
        tam_bloque=10000,
        max_pasos=350,
        eval_intermedia=1000,
        epsilon_min=0.01,
        alpha_offset=cfg['alpha_offset'],
        alpha_min=cfg['alpha_min']
    )
    tiempo_opt = time.time() - t0
    
    # Política final
    acciones_mapa = {0: '←', 1: '↓', 2: '→', 3: '↑'}
    policy_opt = np.array([acciones_mapa[int(np.argmax(agent_opt.qtable[s, :]))] 
                          for s in range(agent_opt.qtable.shape[0])]).reshape(4, 4)
    
    resultado_opt = {
        'nombre': cfg['nombre'],
        'tasa_final': res_opt['tasa_victorias'],
        'recompensa': res_opt['recompensa_promedio'],
        'desv_std': res_opt['desv_estandar'],
        'tiempo_s': tiempo_opt,
        'gamma': cfg['gamma'],
        'alpha_offset': cfg['alpha_offset'],
        'alpha_min': cfg['alpha_min'],
        'politica': policy_opt,
        'progreso': prog_opt
    }
    
    resultados_optimizacion.append(resultado_opt)
    
    print(f"   ✅ Resultado: {res_opt['tasa_victorias']:.1%}")
    print(f"   ⏱️  Tiempo: {tiempo_opt:.1f}s")
    print(f"   🧭 Política final:\n{policy_opt}")

# Análisis de resultados
print("\n" + "="*60)
print("📊 RESULTADOS ")
print("="*60)

df_opt = pd.DataFrame(resultados_optimizacion)
df_opt_sorted = df_opt.sort_values('tasa_final', ascending=False)

mejor_opt = df_opt_sorted.iloc[0]
print(f"\n🏆 MEJOR RESULTADO:")
print(f"   {mejor_opt['nombre']}: {mejor_opt['tasa_final']:.1%}")
print(f"   γ={mejor_opt['gamma']}, α_offset={mejor_opt['alpha_offset']}, α_min={mejor_opt['alpha_min']}")
print(f"   Recompensa: {mejor_opt['recompensa']:.3f} ± {mejor_opt['desv_std']:.3f}")


# Gráfico de progreso
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
for i, res in enumerate(resultados_optimizacion):
    prog = res['progreso']
    plt.plot(prog['bloque'] * 15000, prog['tasa'], 
             marker='o', label=res['nombre'], alpha=0.8)
plt.xlabel('Episodios')
plt.ylabel('Tasa greedy')
plt.title('Progreso α=1/visitas optimizado')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
plt.bar(range(len(df_opt)), df_opt['tasa_final'], alpha=0.7)
plt.axhline(y=0.5, color='red', linestyle='--', label='Objetivo 50%')
plt.xticks(range(len(df_opt)), [f"Cfg {i+1}" for i in range(len(df_opt))])
plt.ylabel('Tasa final')
plt.title('Comparación final')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 3)
for i, res in enumerate(resultados_optimizacion):
    prog = res['progreso']
    plt.plot(prog['bloque'] * 15000, prog['alpha_promedio'], 
             marker='.', label=res['nombre'], alpha=0.8)
plt.xlabel('Episodios')
plt.ylabel('α promedio')
plt.title('Evolución de α promedio')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
plt.scatter(df_opt['gamma'], df_opt['tasa_final'], s=150, alpha=0.7)
for i, row in df_opt.iterrows():
    plt.annotate(f"Cfg {i+1}", (row['gamma'], row['tasa_final']), 
                xytext=(5, 5), textcoords='offset points')
plt.xlabel('Gamma')
plt.ylabel('Tasa final')
plt.title('Gamma vs Rendimiento')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📋 TABLA DETALLADA:")
tabla_opt = df_opt_sorted[['nombre', 'tasa_final', 'gamma', 'alpha_offset', 'alpha_min', 'tiempo_s']].round(4)
print(tabla_opt.to_string(index=False))

In [52]:
param_ranges = {
    "tamano_oculto": [128, 256],
    "gamma": [0.90, 0.95, 0.99],
    "epsilon_decay" : [0.85, 0.90, 0.95],
    "lr": [0.0001, 0.00025, 0.0005]
}

In [58]:
# Ejecución del Agente DQN NO Determinista
env = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=True)
agente_dqn = AgenteDQN(4)
agente_dqn, _ = optimizar_hiperparametros(agente_dqn, param_ranges, env)
resultados = evaluar_agente(agente_dqn, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados, "Agente DQN - mejores hiperparámetros")

Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.85, 'lr': 0.0001}
Entrenamiento detenido en 19999 episodios.edio = 0.568, Longitud Promedio = 33.9, Sin mejora = 0
Recompensa promedio: 0.681
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.85, 'lr': 0.00025}
Entrenamiento detenido en 19999 episodios.edio = 0.582, Longitud Promedio = 34.3, Sin mejora = 0
Recompensa promedio: 0.701
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.85, 'lr': 0.0005}
Entrenamiento detenido en 19999 episodios.edio = 0.570, Longitud Promedio = 33.7, Sin mejora = 1
Recompensa promedio: 0.668
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.9, 'lr': 0.0001}
Entrenamiento detenido en 19999 episodios.edio = 0.572, Longitud Promedio = 34.4, Sin mejora = 0
Recompensa promedio: 0.484
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.9, 'lr': 0.00025}
Entr

## 4. **Comparación entre modelos**
Las siguientes celdas presentan la comparación en el desempeño entre los modelos de Q-Learning y DQN.